# 04 — Inference with Fine-Tuned YOLO26

**Goal:** Run inference using the BDD100K fine-tuned YOLO26 model.

This notebook:
1. Loads trained weights
2. Runs inference on validation / sample images
3. Visualises predictions
4. Saves output images
5. Shows how to reuse trained weights

## 1 — Setup

In [ ]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

# ── Paths ──────────────────────────────────────────────────────────
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
YOLO_DATASET_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo")

# Path to trained weights (from notebook 03)
WEIGHTS_PATH = os.path.join(ECOCAR_ROOT, "weights", "bdd100k_yolo26s_best.pt")
# Alternative: directly from training run
# WEIGHTS_PATH = os.path.join(ECOCAR_ROOT, "training_runs", "bdd100k_yolo26s", "weights", "best.pt")

# Output directory
OUTPUT_DIR = os.path.join(ECOCAR_ROOT, "outputs", "04_inference")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Weights:    {WEIGHTS_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Exists:     {os.path.isfile(WEIGHTS_PATH)}")

## 2 — Load Trained Model

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS_PATH)

print(f"\n✅ Model loaded: {WEIGHTS_PATH}")
print(f"   Task: {model.task}")
print(f"   Classes ({len(model.names)}):")
for idx, name in model.names.items():
    print(f"     {idx}: {name}")

## 3 — Select Inference Images

In [ ]:
import random

# Use validation images
val_images_dir = os.path.join(YOLO_DATASET_DIR, "images", "val")

if os.path.isdir(val_images_dir):
    all_images = [os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir)
                  if f.endswith(('.jpg', '.png', '.jpeg'))]
    # Select a random sample
    NUM_SAMPLES = 8
    sample_images = random.sample(all_images, min(NUM_SAMPLES, len(all_images)))
    print(f"✅ Selected {len(sample_images)} images from {len(all_images)} val images")
else:
    print(f"❌ Val images not found at {val_images_dir}")
    print("   Falling back to sample images...")
    
    import urllib.request
    sample_dir = "sample_images"
    os.makedirs(sample_dir, exist_ok=True)
    urls = {
        "bus.jpg": "https://ultralytics.com/images/bus.jpg",
        "zidane.jpg": "https://ultralytics.com/images/zidane.jpg",
    }
    for name, url in urls.items():
        path = os.path.join(sample_dir, name)
        if not os.path.exists(path):
            urllib.request.urlretrieve(url, path)
    sample_images = [os.path.join(sample_dir, f) for f in os.listdir(sample_dir)]

## 4 — Run Inference

In [ ]:
# Run inference on all sample images
results = model(sample_images, verbose=False)

print(f"✅ Inference complete on {len(results)} images")

## 5 — Visualise Predictions

In [ ]:
# Display predictions in a grid
n_cols = 2
n_rows = (len(results) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :] if n_cols > 1 else np.array([[axes]])
if n_cols == 1:
    axes = axes[:, np.newaxis]

for i, result in enumerate(results):
    row, col = i // n_cols, i % n_cols
    annotated = result.plot()
    axes[row, col].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[row, col].set_title(f"{os.path.basename(sample_images[i])} — {len(result.boxes)} detections", fontsize=9)
    axes[row, col].axis('off')

# Hide empty subplots
for i in range(len(results), n_rows * n_cols):
    row, col = i // n_cols, i % n_cols
    axes[row, col].axis('off')

plt.suptitle('BDD100K Fine-Tuned YOLO26 — Predictions', fontsize=14)
plt.tight_layout()
plt.show()

## 6 — Save Output Images

In [ ]:
for i, result in enumerate(results):
    annotated = result.plot()
    save_path = os.path.join(OUTPUT_DIR, f"pred_{os.path.basename(sample_images[i])}")
    cv2.imwrite(save_path, annotated)
    print(f"💾 {save_path}")

print(f"\n✅ All predictions saved to: {OUTPUT_DIR}")

## 7 — Detailed Prediction Info

In [ ]:
for i, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Image: {os.path.basename(sample_images[i])}")
    print(f"{'='*60}")
    
    boxes = result.boxes
    if len(boxes) == 0:
        print("  No detections.")
        continue
    
    # Summary by class
    class_counts = {}
    for box in boxes:
        cls_id = int(box.cls.item())
        cls_name = model.names[cls_id]
        class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
    
    print(f"  Detections by class:")
    for name, count in sorted(class_counts.items(), key=lambda x: -x[1]):
        print(f"    {name}: {count}")
    
    # Detailed boxes
    print(f"\n  {'Class':<20} {'Conf':>6} {'Box (x1,y1,x2,y2)'}")
    print(f"  {'-'*55}")
    for box in boxes:
        cls_id = int(box.cls.item())
        cls_name = model.names[cls_id]
        conf = box.conf.item()
        coords = box.xyxy[0].cpu().numpy().astype(int)
        print(f"  {cls_name:<20} {conf:>6.3f} ({coords[0]}, {coords[1]}, {coords[2]}, {coords[3]})")

## 8 — Side-by-Side: Pretrained vs Fine-Tuned

Compare the COCO-pretrained baseline with the BDD100K fine-tuned model.

In [ ]:
# Load the pretrained COCO model for comparison
pretrained_model = YOLO("yolo26n.pt")  # COCO pretrained

# Pick first 2 images for comparison
compare_images = sample_images[:2]

for img_path in compare_images:
    # Pretrained results
    pre_result = pretrained_model(img_path, verbose=False)[0]
    pre_annotated = pre_result.plot()
    
    # Fine-tuned results
    ft_result = model(img_path, verbose=False)[0]
    ft_annotated = ft_result.plot()
    
    # Side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    ax1.imshow(cv2.cvtColor(pre_annotated, cv2.COLOR_BGR2RGB))
    ax1.set_title(f"COCO Pretrained ({len(pre_result.boxes)} dets)", fontsize=12)
    ax1.axis('off')
    ax2.imshow(cv2.cvtColor(ft_annotated, cv2.COLOR_BGR2RGB))
    ax2.set_title(f"BDD100K Fine-Tuned ({len(ft_result.boxes)} dets)", fontsize=12)
    ax2.axis('off')
    plt.suptitle(os.path.basename(img_path), fontsize=13)
    plt.tight_layout()
    
    # Save comparison
    save_path = os.path.join(OUTPUT_DIR, f"compare_{os.path.basename(img_path)}")
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"💾 {save_path}")

## 9 — How to Reuse Trained Weights

```python
# In any notebook or script:
from ultralytics import YOLO

# Load your fine-tuned model
model = YOLO("/content/drive/MyDrive/EcoCAR/weights/bdd100k_yolo26s_best.pt")

# Run inference
results = model("path/to/image.jpg")

# Access predictions
for result in results:
    boxes = result.boxes
    for box in boxes:
        class_name = model.names[int(box.cls)]
        confidence = box.conf.item()
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
```

In [ ]:
print("\n" + "="*60)
print(" INFERENCE SUMMARY")
print("="*60)
print(f" Model:       {WEIGHTS_PATH}")
print(f" Images:      {len(results)}")
total_dets = sum(len(r.boxes) for r in results)
print(f" Detections:  {total_dets}")
print(f" Saved to:    {OUTPUT_DIR}")
print("="*60)
print("\n🎯 Next: notebook 05 (feature extraction) or 06 (model inspection)")